This is a notebook for developing a method for batch downloading CRISM VRDR products and resampling them to CaSSIS wavelengths. We will also resample to PanCam and OROCHI wavelengths. We could also do Mastcam, Mastcam-Z, HiRISE, HRSC whilst we're here.

# Overview

1. Get a list of VRDR tiles
2. Download a VRDR Tile
3. Access the CaSSIS transmission profiles
4. Apply the transmission profiles to the VRDR tile
5. Save the CaSSIS band simulated image
6. Batch Process the list of all VRDR Tiles

# 1. Getting the VRDR Tile List

The footprint file is found at: https://ode.rsl.wustl.edu/mars/datafile/derived_products/coverageshapefiles/mars/mro/crism/vrdr/mars_mro_crism_vrdr_c0a.zip

In [ ]:
from pathlib import Path
import geopandas as gpd

In [ ]:
import requests
import zipfile

# Download and extract the CRISM VRDR footprint shapefile
def download_and_extract_footprint():
    """Download and extract the CRISM VRDR footprint shapefile"""
    
    # URL for the CRISM VRDR footprint shapefile
    footprint_url = "https://ode.rsl.wustl.edu/mars/datafile/derived_products/coverageshapefiles/mars/mro/crism/vrdr/mars_mro_crism_vrdr_c0a.zip"
    
    # Create the directory if it doesn't exist
    crism_footprint_dir = Path('.', 'crism_vrdr_footprints')
    crism_footprint_dir.mkdir(parents=True, exist_ok=True)
    
    # Download the zip file
    zip_path = crism_footprint_dir / "mars_mro_crism_vrdr_c0a.zip"
    
    if not zip_path.exists():
        print(f"Downloading CRISM VRDR footprint from: {footprint_url}")
        response = requests.get(footprint_url)
        response.raise_for_status()
        
        with open(zip_path, 'wb') as f:
            f.write(response.content)
        print(f"Downloaded to: {zip_path}")
    else:
        print(f"Zip file already exists: {zip_path}")
    
    # Extract the zip file
    shp_path = crism_footprint_dir / "mars_mro_crism_vrdr_c0a.shp"
    if not shp_path.exists():
        print("Extracting shapefile...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(crism_footprint_dir)
        print(f"Extracted shapefile to: {crism_footprint_dir}")
    else:
        print(f"Shapefile already exists: {shp_path}")
    
    return shp_path

In [ ]:
# Download and extract the footprint data
footprint_shp_path = download_and_extract_footprint()

In [ ]:
crism_footprint_dir = Path('.', 'crism_vrdr_footprints')
crism_vrdr_footprint_shp_path = crism_footprint_dir / 'mars_mro_crism_vrdr_c0a.shp'
crism_vrdr_footprint_gdf = gpd.read_file(crism_vrdr_footprint_shp_path)
crism_vrdr_footprint_gdf.head()

We need to filter the dataset down to only Lambert Albedo products:

In [ ]:
# get only rows of the GeoDataFrame where the ProductId contains 'vrral'
crism_vrdr_footprint_gdf = crism_vrdr_footprint_gdf[crism_vrdr_footprint_gdf['ProductId'].str.contains('vrral')]
crism_vrdr_footprint_gdf.head()

We get the file URLs:

In [ ]:
vrdr_file_list = crism_vrdr_footprint_gdf.FilesURL.unique()

In [ ]:
len(vrdr_file_list)

# 2. Download a VRDR Tile

In [ ]:
# download available files at the given URL
import os
import shutil
from typing import Union
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

def download_file(url: str, dest_folder: Union[str, Path], overwrite: bool=False) -> None:
    """Download a file from a URL to a specified destination folder.

    :param url: The URL of the file to download.
    :type url: str
    :param dest_folder: The folder where the file will be saved.
    :type dest_folder: Union[str, Path]
    """
    if not Path(dest_folder).exists():
        Path(dest_folder).mkdir(parents=True, exist_ok=True)

    filename = url.split("/")[-1]
    file_path = Path(dest_folder) / filename

    # Get the page content and parse for downloadable file links
    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    # Find all links to files (e.g., .img, .lbl, .zip, etc.)
    file_links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        # Only consider links that look like downloadable files
        if any(href.lower().endswith(ext) for ext in [".img", ".lbl", ".hdr", ".cat", ".txt", ".png"]):
            # if href contains vrral:
            if 'vrral' not in href:
                continue            
            # If the link is relative, prepend the base URL
            if not href.startswith("http"):
                base_url = url.rsplit("/", 1)[0]
                href = f"{base_url}/{href.lstrip('/')}"
            print(f"Found file link: {href}")
            file_links.append(href)

    if not file_links:
        print(f"No downloadable files found at {url}")
        return file_path

    for file_url in file_links:
        print(f"Downloading {file_url}...")
        filename = file_url.split("/")[-1]
        # get the file stem
        file_stem = filename.split(".")[0]
        product_dir = Path(dest_folder) / file_stem
        product_dir.mkdir(parents=True, exist_ok=True)
        file_path = product_dir / filename

        # check if the file is already downloaded
        if file_path.exists() and not overwrite:
            print(f"{filename} already exists. Skipping download.")
            continue

        with requests.get(file_url, stream=True) as r:

            # check header to get content length
            total_length = int(r.headers.get("Content-Length"))

            # implement progress bar via tqdm
            with tqdm.wrapattr(r.raw, "read", total=total_length, desc="")as raw:
            # save the output to a file
                with open(file_path, 'wb') as f:
                    f.write(r.content)                

        # file_resp = requests.get(file_url)
        # file_resp.raise_for_status()
        # with open(file_path, 'wb') as f:
        print(f"Downloaded {filename} to {file_path}")        
    
    return file_path

In [ ]:
crism_vrdr_dir = Path('.', 'crism_vrdr')
crism_vrdr_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
vrdr_file_list

To test, we have this example:
t0604_vrral_25s088_0654_1

In [ ]:
# find the entry that matches: t1375_vrral_25n343_0654_1
for entry in vrdr_file_list:
    if 't1375_vrral_25n343_0654_1' in entry:
        print(entry)
        break
entry

In [ ]:
file_path = download_file(entry, crism_vrdr_dir)

In [ ]:
file_path

Single vrral cube took 90 mins to download.

In [ ]:
days = (1.5 * 936)/24
print(f"Estimated download time: {days:.2f} days")

# 3. Access a VRDR Tile and Load

In [ ]:
file_path

In [ ]:
import pdr
vrral = pdr.read(file_path)
band_names = vrral.metaget('BAND_NAME')
crism_vrdr_wvls = [float(band) for band in band_names]
vrral.load('IMAGE')
crism_vrdr_cube = vrral.IMAGE
type(crism_vrdr_cube)

# 3. Access the CaSSIS Transmission Profiles

In [ ]:
# read the CaSSIS-2019.csv file
import pandas as pd

def load_cassis_channels() -> pd.DataFrame:
    """Load the CaSSIS channel information from the CSV file."""
    cassis_csv_path = Path('.', 'CaSSIS-2019.csv')
    if not cassis_csv_path.exists():
        raise FileNotFoundError(f"CaSSIS CSV file not found: {cassis_csv_path}")
    
    cassis_df = pd.read_csv(cassis_csv_path)

    # index = cassis_df['wvls']
    cassis_df = cassis_df.set_index('wvls')

    return cassis_df

In [ ]:
cassis_df = load_cassis_channels()

In [ ]:
cassis_df.plot()

# 4. Apply the Transmission Profiles to the VRDR Tile

In [ ]:
from typing import Literal
import numpy as np
from scipy.interpolate import interp1d

def apply_cassis_band(
        cassis_band: Literal['BLU', 'PAN', 'RED', 'NIR'], 
        cassis_df: pd.DataFrame,
        crism_vrdr_wvls: list, 
        crism_vrdr_cube: np.ndarray) -> np.ndarray:
    # get the selected cassis band transmission profile
    band_profile = cassis_df[cassis_band].values
    profile_wvls = cassis_df.index

    interp_func = interp1d(profile_wvls, band_profile, kind='linear', bounds_error=False, fill_value='extrapolate')
    resampled_profile = interp_func(crism_vrdr_wvls)
    # clip
    resampled_profile = np.clip(resampled_profile, 0, None)

    # set max values to nan
    crism_vrdr_cube[crism_vrdr_cube == crism_vrdr_cube.max()] = np.nan

    # efficiently mulitply the CRISM VRDR cube by the resampled CaSSIS profile
    cassis_band_cube = crism_vrdr_cube * resampled_profile[:, np.newaxis, np.newaxis]
    # efficiently sum the cube along the spectral axis, and divide by the sum of the resampled profile to normalise
    cassis_band_image = np.nansum(cassis_band_cube, axis=0) / np.sum(resampled_profile)

    return cassis_band_image

In [ ]:
def apply_stretch(image: np.ndarray, lower_percentile: float=0.01, upper_percentile: float=99.95) -> np.ndarray:
    """Apply a linear stretch to the image based on the given percentiles."""
    # treat 0 values as NaN
    image = image.astype(float)
    image[image == 0] = np.nan
    lower = np.nanpercentile(image, lower_percentile)
    upper = np.nanpercentile(image, upper_percentile)
    print(f"Stretching with lower={lower:.4f}, upper={upper:.4f}")
    stretched = (image - lower) / (upper - lower)
    stretched = np.clip(stretched, 0.0, 1.0)
    
    return stretched

In [ ]:
cassis_cube = []
cassis_stretch_cube = []
for band in ['BLU', 'PAN', 'RED', 'NIR']:
    cassis_band_image = apply_cassis_band(band, cassis_df, crism_vrdr_wvls, crism_vrdr_cube)
    cassis_cube.append(cassis_band_image) 
    print(f"Processing band: {band}")   
    cassis_stretch_image = apply_stretch(cassis_band_image)
    cassis_stretch_cube.append(cassis_stretch_image)

In [ ]:
# make browse product
def cassis_browse_product(
        product: Literal['NPB', 'NRB', 'NRP', 'RPB', 'RGB'],
        cassis_cube,
        cassis_stretch_cube
    ) -> np.array:

    if product == 'NPB':
        browse_arr = np.array([cassis_stretch_cube[3], cassis_stretch_cube[1], cassis_stretch_cube[0]])

    if product == 'NRB':
        browse_arr = np.array([cassis_stretch_cube[3], cassis_stretch_cube[2], cassis_stretch_cube[0]])

    if product == 'NRP':
        browse_arr = np.array([cassis_stretch_cube[3], cassis_stretch_cube[2], cassis_stretch_cube[1]])
    
    if product == 'RPB':
        browse_arr = np.array([cassis_stretch_cube[2], cassis_stretch_cube[1], cassis_stretch_cube[0]])

    if product == 'RGB':
        # make synthetic blu channel
        sBLU = (2*cassis_cube[0]) - (0.3*cassis_cube[1])
        sBLU_stretch = apply_stretch(sBLU)
        browse_arr = np.array([cassis_stretch_cube[1], cassis_stretch_cube[0], sBLU_stretch])

    browse_arr = browse_arr.transpose((1, 2, 0))

    return browse_arr

In [ ]:
np.mean(cassis_cube, axis=0).shape

In [ ]:
from typing import Tuple

def cassis_spectral_parameters(
        parameter: Literal[
            'PBR', #PAN/BLU
            'PNR', #PAN/NIR
            'RPR', #RED/PAN
            'ICE', #MEAN[B,P,R,N]/(NIR/BLU)
            'CBRC1', #(RPR, PBR, PNR)->(R,G,B)
            'CBRC2', #(RPR, PBR, ICE)->(R,G,B)
        ],
        cassis_cube        
    )->Tuple[np.array,np.array]:
    
    bands=1
    if parameter == 'PBR':
        sp_arr = cassis_cube[1] / cassis_cube[0]

    if parameter == 'PNR':
        sp_arr = cassis_cube[1] / cassis_cube[3]

    if parameter == 'RPR':
        sp_arr = cassis_cube[2] / cassis_cube[1]
    
    if parameter == 'ICE':
        sp_arr = np.mean(cassis_cube, axis=0) / (cassis_cube[3] / cassis_cube[0])
    
    if parameter == 'CBRC1':
        bands = 3
        sp_arr = np.array([
            cassis_cube[2] / cassis_cube[1],
            cassis_cube[1] / cassis_cube[0],
            cassis_cube[1] / cassis_cube[3]
        ])
        sp_arr = sp_arr.transpose((1, 2, 0))
    
    if parameter == 'CBRC2':
        bands = 3
        sp_arr = np.array([
            cassis_cube[2] / cassis_cube[1],
            cassis_cube[1] / cassis_cube[0],
            np.mean(cassis_cube, axis=0) / (cassis_cube[3] / cassis_cube[0])
        ])
        sp_arr = sp_arr.transpose((1, 2, 0))

    # for each sp_arr channel, apply_stretch
    if bands == 1:
        sp_stretch = apply_stretch(sp_arr)
    if bands == 3:
        sp_stretch = np.array([apply_stretch(sp_arr[:,:,band]) for band in range(bands)])
        sp_stretch = sp_stretch.transpose((1,2,0))

    return sp_arr, sp_stretch


In [ ]:
npb = cassis_browse_product('NPB', cassis_cube, cassis_stretch_cube)

In [ ]:
rgb = cassis_browse_product('RGB', cassis_cube, cassis_stretch_cube)

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
pbr,pbr_stretch = cassis_spectral_parameters('PBR', cassis_cube)
plt.imshow(pbr_stretch, cmap='gray')

In [ ]:
pnr,pnr_stretch = cassis_spectral_parameters('PNR', cassis_cube)
plt.imshow(pnr_stretch, cmap='gray')

In [ ]:
rpr,rpr_stretch = cassis_spectral_parameters('RPR', cassis_cube)
plt.imshow(rpr_stretch, cmap='gray')

In [ ]:
ice,ice_stretch = cassis_spectral_parameters('ICE', cassis_cube)
plt.imshow(ice_stretch, cmap='gray')

In [ ]:
cbrc1,cbrc1_stretch = cassis_spectral_parameters('CBRC1', cassis_cube)
plt.imshow(cbrc1_stretch, interpolation=None)

In [ ]:
cbrc2,cbrc2_stretch = cassis_spectral_parameters('CBRC2', cassis_cube)
plt.imshow(cbrc2_stretch)

In [ ]:
# npb histogram
plt.hist(npb.flatten(), bins=100)
plt.xlabel('I/F')
plt.ylabel('Count')
plt.title('Histogram of CaSSIS RGB Image Values')

In [ ]:
plt.imshow(npb)

In [ ]:
plt.imshow(rgb)

# 5. Export the CaSSIS Simulated Tile

In [ ]:
import spectral.io.envi as envi

In [ ]:
# get the envi hdr file for the give tile
import pdr
vrral = pdr.read(file_path)
band_names = vrral.metaget('BAND_NAME')
crism_vrdr_wvls = [float(band) for band in band_names]
vrral.load('ENVI_HEADER')
vrral.ENVI_HEADER

In [ ]:
vrral.metadata

In [ ]:
# change extension of file path from lbl to hdr
envi_header_path = file_path.with_suffix('.hdr')

In [ ]:
envi.open(envi_header_path)

In [ ]:
vrral.metaget

In [ ]:
output_dir = Path('cassis', file_path.parent.stem)
output_dir.mkdir(parents=True, exist_ok=True)
output_envi_hdr_path = output_dir / f"{file_path.stem}_cassis.hdr"

In [ ]:
cassis_arr = np.array(cassis_cube)

In [ ]:
cassis_arr = cassis_arr.transpose((1,2,0))

In [ ]:
vrral.metaget('LINES')

In [ ]:
metadata = {
    'lines': vrral.metaget('LINES'),
    'samples': vrral.metaget('LINE_SAMPLES'),
    'bands': 4,
    'wavelengths': [499, 675, 836, 937]
}
img = envi.create_image(output_envi_hdr_path, metadata, dtype=np.float32, shape=cassis_arr.shape, force=True)

In [ ]:
mm = img.open_memmap(writable=True)

In [ ]:
mm = cassis_arr

In [ ]:
envi.save_image(output_envi_hdr_path, cassis_arr,force=True)

# 6. Batch Process for all Tiles